# 01 — Data Inspection and Exploratory Data Analysis

This notebook performs the data-integrity and exploratory analysis directly from the source CSV.

It covers:

- source-data structure and target balance;
- conventional missing values and explicit `unknown` categories;
- exact duplicates and pseudo-repeat-profile sensitivity;
- numeric and categorical summaries;
- contiguous ordered month-labelled blocks;
- pre-call Pearson and bias-corrected Cramér's V associations;
- a separate post-call `duration` leakage diagnostic;
- pre-call numeric inter-feature correlations;
- the exploratory higher-conversion customer population.

`duration` is deliberately excluded from the deployable/pre-call association set because it is only known after a call. Its association with subscription is reported separately as a leakage diagnostic. All values are calculated from the source CSV during execution.
This clean rerun version has no dependency on `data/`, `results/`, or `figures/` folders and does not write CSV/PNG artifacts during execution.


## 1. Setup and reusable analysis functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, norm
from IPython.display import display

DATA_SOURCE='term-deposit-marketing-2020-labelled.csv'

TARGET='y'
PROFILE=['age','job','marital','education','balance']

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)


def wilson(s,n,conf=.95):
    if n==0:
        return np.nan,np.nan
    z=norm.ppf(1-(1-conf)/2)
    p=s/n
    d=1+z*z/n
    c=p+z*z/(2*n)
    a=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return (c-a)/d,(c+a)/d

def cramer(x,y):
    t=pd.crosstab(x,y)
    n=t.to_numpy().sum()
    if t.empty or n<=1:
        return np.nan
    chi=chi2_contingency(t,correction=False)[0]
    r,k=t.shape
    phi=chi/n
    pc=max(0,phi-((k-1)*(r-1))/(n-1))
    rc=r-((r-1)**2)/(n-1)
    kc=k-((k-1)**2)/(n-1)
    d=min(rc-1,kc-1)
    return np.sqrt(pc/d) if d>0 else 0.

def assoc_num(d,fs):
    z=d[fs].corrwith(d.y_binary).rename('association').reset_index().rename(columns={'index':'feature'})
    z['variable_type']='numeric'
    z['measure']='Pearson r'
    z['association_strength']=z.association.abs()
    return z.sort_values('association_strength',ascending=False).reset_index(drop=True)

def assoc_cat(d,fs):
    z=pd.DataFrame({'feature':fs,'association':[cramer(d[f],d[TARGET]) for f in fs]})
    z['variable_type']='categorical'
    z['measure']="Cramer's V"
    z['association_strength']=z.association
    return z.sort_values('association_strength',ascending=False).reset_index(drop=True)

def conv(d,f):
    z=d.groupby(f,dropna=False).y_binary.agg(customers='size',subscribers='sum').reset_index()
    z['subscription_rate']=z.subscribers/z.customers
    ci=z.apply(lambda r:wilson(int(r.subscribers),int(r.customers)),axis=1)
    z[['ci_lower','ci_upper']]=pd.DataFrame(ci.tolist(),index=z.index)
    return z.sort_values('subscription_rate',ascending=False).reset_index(drop=True)

def blocks(d):
    z=d.copy()
    z['row_order']=np.arange(len(z))
    z['period_id']=z.month.astype(str).ne(z.month.astype(str).shift()).cumsum()
    r=z.groupby('period_id').agg(
        month=('month','first'),
        start_row=('row_order','min'),
        end_row=('row_order','max'),
        customers=('y_binary','size'),
        subscribers=('y_binary','sum')
    ).reset_index()
    r['subscription_rate']=r.subscribers/r.customers
    return r

## 2. Load the dataset and extract its structure

In [ ]:
df=pd.read_csv(DATA_SOURCE)
if 'y_binary' in df.columns:
    df=df.drop(columns='y_binary')
df['y_binary']=df[TARGET].eq('yes').astype(int)

source=[c for c in df.columns if c!='y_binary']
predictors=[c for c in source if c!=TARGET]
numeric=df[predictors].select_dtypes(include=np.number).columns.tolist()
categorical=[c for c in predictors if c not in numeric]

n=len(df)
yes=int(df.y_binary.sum())
rate=df.y_binary.mean()
majority=df[TARGET].value_counts(normalize=True).max()

columns=pd.DataFrame({
    'variable':source,
    'dtype':df[source].dtypes.astype(str).values,
    'unique_values':[df[c].nunique(dropna=False) for c in source]
})

target=df[TARGET].value_counts().rename('count').to_frame()
target['percentage']=100*target['count']/n

dataset_summary=pd.DataFrame({
    'measure':['rows','source_columns','predictors','numeric_predictors','categorical_predictors',
               'subscribers','non_subscribers','subscription_rate','majority_accuracy'],
    'value':[n,len(source),len(predictors),len(numeric),len(categorical),yes,n-yes,rate,majority]
})

display(df[source].head())
display(columns)
display(target.round(2))
display(dataset_summary.round(4))


## 3. Data quality, explicit unknowns and pseudo-repeat sensitivity

In [ ]:
missing=df[source].isna().sum().rename('missing_count').to_frame()
missing['missing_percentage']=100*missing.missing_count/n

# Distinguish genuine missing values, blank strings, common text placeholders, and explicit 'unknown'.
obj_cols=df[source].select_dtypes(include='object').columns.tolist()
blank_counts=pd.Series({c: df[c].astype(str).str.strip().eq('').sum() for c in obj_cols}, name='blank_count')
placeholder_tokens={'na','n/a','none','null','nan','missing'}
placeholder_counts=pd.Series({
    c: df[c].astype(str).str.strip().str.lower().isin(placeholder_tokens).sum() for c in obj_cols
}, name='text_placeholder_count')
missing_audit=pd.DataFrame(index=source)
missing_audit['genuine_missing_count']=df[source].isna().sum()
missing_audit['blank_string_count']=blank_counts.reindex(source, fill_value=0)
missing_audit['text_placeholder_count']=placeholder_counts.reindex(source, fill_value=0)
exact_dup=int(df[source].duplicated().sum())

unknown=pd.DataFrame({
    'feature':categorical,
    'unknown_count':[df[f].astype(str).str.strip().str.lower().eq('unknown').sum() for f in categorical]
})
unknown['unknown_percentage']=100*unknown.unknown_count/n
unknown=unknown.sort_values('unknown_count',ascending=False).reset_index(drop=True)

profile_id=df.groupby(PROFILE,dropna=False,sort=False).ngroup()
ps=pd.Series(profile_id).value_counts()
repeat_rows=int(ps[ps>1].sum())
repeat_groups=int((ps>1).sum())
tmp=df.assign(profile_id=profile_id)
month_span=tmp.groupby('profile_id').month.nunique()

repeat_profile=pd.DataFrame([{
    'profile_fields':' + '.join(PROFILE),
    'repeat_rows':repeat_rows,
    'repeat_row_rate':repeat_rows/n,
    'repeat_groups':repeat_groups,
    'groups_spanning_2_month_labels':int((month_span==2).sum()),
    'groups_spanning_gt1_month_label':int((month_span>1).sum()),
    'interpretation':'pseudo-profile sensitivity only; no true customer ID'
}])

display(missing.round(4))
print('Missingness audit by representation:')
display(missing_audit)
print(f"Exact duplicate rows: {exact_dup:,}")
display(unknown.round(4))
display(repeat_profile)


## 4. Target balance and explicit unknown categories

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
p=target.reset_index(names='outcome')
bars=ax.bar(p.outcome,p['count'])
ax.set(title='Term-Deposit Subscription Outcome',ylabel='Customers')
for b,c,pct in zip(bars,p['count'],p['percentage']):
    ax.text(b.get_x()+b.get_width()/2,b.get_height(),f'{c:,}\n({pct:.2f}%)',ha='center',va='bottom')
plt.tight_layout()
plt.show()

up=unknown[unknown.unknown_count>0]
if len(up):
    fig,ax=plt.subplots(figsize=(7,4))
    bars=ax.bar(up.feature,up.unknown_percentage)
    ax.set(title="Explicit 'unknown' Categories",ylabel='Share of records (%)')
    for b,v in zip(bars,up.unknown_percentage):
        ax.text(b.get_x()+b.get_width()/2,b.get_height(),f'{v:.2f}%',ha='center',va='bottom')
    plt.tight_layout()
plt.show()

## 5. Numeric summaries and categorical conversion tables

In [ ]:
num_summary=df[numeric].describe().T
num_summary['skewness']=df[numeric].skew()
display(num_summary.round(3))
cat_tables={}
for f in categorical:
    cat_tables[f]=conv(df,f)
for f in ['contact','job','education','month']:
    print(f"\nSubscription outcomes by {f}:")
    display(cat_tables[f].assign(
        subscription_rate_pct=100*cat_tables[f].subscription_rate,
        ci_lower_pct=100*cat_tables[f].ci_lower,
        ci_upper_pct=100*cat_tables[f].ci_upper
    )[[f,'customers','subscribers','subscription_rate_pct','ci_lower_pct','ci_upper_pct']].round(2))

## 6. Ordered contiguous month-labelled blocks

In [ ]:
periods=blocks(df)
display(periods.assign(subscription_rate_pct=100*periods.subscription_rate).round(3))
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(periods.period_id,100*periods.subscription_rate,marker='o')
ax.set(
    title='Subscription Rate Across Ordered Month-Labelled Blocks',
    xlabel='Ordered source-file block',
    ylabel='Subscription rate (%)'
)
plt.tight_layout()
plt.show()

## 7. Pre-call associations, duration leakage diagnostic and numeric inter-feature correlations

The deployable association analysis contains only variables available in the pre-call feature set. `duration` is post-call information, so it is excluded from the pre-call Pearson table and figure and evaluated separately as a leakage diagnostic.

In [ ]:
pre_num=[f for f in numeric if f!='duration']
pearson=assoc_num(df,pre_num)
duration_diag=assoc_num(df,['duration'])
cramers=assoc_cat(df,categorical)
all_assoc=pd.concat([pearson,cramers],ignore_index=True)

# Canonical association outputs: deployable/pre-call predictors only.
corr=df[pre_num].corr()
upper=corr.where(np.triu(np.ones(corr.shape),1).astype(bool))
pairs=upper.stack().rename('correlation').reset_index().rename(columns={'level_0':'feature_1','level_1':'feature_2'})
pairs['absolute_correlation']=pairs.correlation.abs()
pairs=pairs.sort_values('absolute_correlation',ascending=False).reset_index(drop=True)
print("Pre-call numeric feature-to-target associations:")
display(pearson.round(4))
print("Post-call duration leakage diagnostic:")
display(duration_diag.round(4))
print("Categorical pre-call feature-to-target associations:")
display(cramers.round(4))
print("Pre-call numeric correlation matrix:")
display(corr.round(3))
print("Strongest pre-call numeric feature pairs:")
display(pairs.head(10).round(4))

fig,axes=plt.subplots(1,2,figsize=(12,5))
q=pearson.sort_values('association')
axes[0].barh(q.feature,q.association)
axes[0].axvline(0)
axes[0].set(title='Pre-Call Numeric Association With Subscription',xlabel='Pearson r')

q=cramers.sort_values('association')
axes[1].barh(q.feature,q.association)
axes[1].set(title='Categorical Association With Subscription',xlabel="Cramer's V")

plt.tight_layout()
plt.show()

# Duration is intentionally isolated so it cannot be mistaken for a deployable predictor.
fig,ax=plt.subplots(figsize=(7,2.8))
q=duration_diag.sort_values('association')
ax.barh(q.feature,q.association)
ax.axvline(0)
ax.set(title='Post-Call Duration Leakage Diagnostic',xlabel='Pearson r')
plt.tight_layout()
plt.show()


## 8. Exploratory higher-conversion population

In [ ]:
q3=df.balance.quantile(.75)
seg=(
    df.contact.astype(str).str.lower().eq('cellular')
    & (
        df.job.astype(str).str.lower().eq('retired')
        | df.age.ge(71)
        | df.balance.ge(q3)
    )
)

seg_n=int(seg.sum())
seg_y=int(df.loc[seg,'y_binary'].sum())

segment=pd.DataFrame([{
    'balance_q3':q3,
    'age_threshold':71,
    'customers':seg_n,
    'subscribers':seg_y,
    'subscription_rate':seg_y/seg_n,
    'full_rate':rate,
    'relative_uplift':(seg_y/seg_n)/rate-1,
    'status':'exploratory data-derived rule; not causal or pre-registered'
}])

display(segment.round(4))


## 9. Consolidated EDA summary

In [ ]:
summary=pd.DataFrame({
    'metric':[
        'rows','subscribers','subscription_rate','majority_accuracy','exact_duplicate_rows',
        'pseudo_repeat_rows','pseudo_repeat_rate','largest_unknown_feature','largest_unknown_rate',
        'strongest_pre_call_numeric_feature','strongest_pre_call_numeric_association',
        'duration_post_call_pearson_diagnostic',
        'strongest_categorical_feature','strongest_categorical_association',
        'strongest_numeric_pair','strongest_numeric_pair_correlation',
        'ordered_blocks','segment_customers','segment_subscribers','segment_rate'
    ],
    'value':[
        n,yes,rate,majority,exact_dup,repeat_rows,repeat_rows/n,
        unknown.iloc[0].feature,unknown.iloc[0].unknown_percentage/100,
        pearson.iloc[0].feature,pearson.iloc[0].association,
        duration_diag.iloc[0].association,
        cramers.iloc[0].feature,cramers.iloc[0].association,
        f"{pairs.iloc[0].feature_1} ↔ {pairs.iloc[0].feature_2}",
        pairs.iloc[0].correlation,len(periods),seg_n,seg_y,seg_y/seg_n
    ]
})

display(summary)
